# Lecture 13 - Le Poisson

McElreath's lecture for today: https://www.youtube.com/watch?v=YrwL6t0kW2I&list=PLDcUM9US4XdMROZ57-OIRtIK0aOynbgZN&index=11

McElreath's lectures for the whole book are available here: https://github.com/rmcelreath/stat_rethinking_2022

An R/Stan repo of code is available here: https://vincentarelbundock.github.io/rethinking2/

An excellent port to Python/PyMC Code is available here: https://github.com/dustinstansbury/statistical-rethinking-2023

You are encouraged to work through both of these versions to re-enforce what we're doing in class.

In [ ]:
# Load R packages
library(cmdstanr)    # R interface to Stan
library(posterior)   # Working with posterior draws
library(bayesplot)   # Plotting posterior draws
library(loo)         # WAIC and PSIS-LOO cross-validation

# Helper functions
stdize <- function(x) (x - mean(x))/sd(x)

indexall <- function(L){
    poo <- unique(L)
    Ix <- as.integer(factor(L, levels = poo))
    list(poo, Ix)
}

# Stan's sampler with the same defaults as pm.sample(1000): 4 chains, 1000 warmup, 1000 draws
sample_stan <- function(model, data){
    model$sample(data = data, chains = 4, parallel_chains = 4, iter_warmup = 1000,
                 iter_sampling = 1000, refresh = 0, show_exceptions = FALSE)
}

# Save the current figure to file (uncomment the savefig() calls below to use)
savefig <- function(file, width = 7, height = 5){
    dev.copy(jpeg, file, width = width, height = height, units = "in", res = 300)
    invisible(dev.off())
}

## Oceanic tool complexity

A fascinating set of data from [Michelle Kline](https://www.michelleakline.com/), whereby she collected information about the size and complexity of tools among polyensian and melanesian societies. The idea is that larger, more connected populations should have more complex sets of tools. So let's take a look and see:

In [ ]:
# Grab tool data
kdata <- read.csv('Kline.csv', sep = ';')
kdata

And this is the entire dataset - this should give us pause about sample sizes and why McElreath is so focused on regularization, but also give us hope that we can make inferences from data in a Bayesian context by thinking carefully about what we're doing. 

Here we want to know: does the number of tools increase with log(population) size and the contact rate among islands? To do this, we can build a Poisson model for tool counts:

$$
\begin{align}
T_i \sim & Poisson(\lambda_i) \\
log(\lambda_i) = & \beta_c + \beta_p log(P)
\end{align}
$$

where there are distinct interepts $\beta_c$ and population size effects $\beta_b$ for high and low contact islands. Given how small this dataset is, priors become very important. Let's simulate first some intercepts to see what the Poisson and log link do to our numbers.

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4)
# Plot priors from N(0, 10)
plot(density(rlnorm(1000, 0, 10)), col = "dodgerblue", lwd = 2,
     xlab = "mean number of tools", ylab = "Density", main = "exp(Bc)")
legend("topright", legend = "Bc ~ LN(0, 10)", col = "dodgerblue", lwd = 2, bty = "n")
# savefig('ln10.jpg', width = 6, height = 4)

Wow, those numbers get big fast - how big? Well the mean value is

In [ ]:
mean(exp(rnorm(10000, 0, 10)))

a boatload. Picking something sensible can be hard. If we eyeball the total tools column stuff is in the 10's (between 13 and 71), so let's try some alternatives:

In [ ]:
mean(exp(rnorm(10000, 2, 2)))

Not bad, let's see what it looks like

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4)
# Plot priors from N(2, 2)
plot(density(rlnorm(10000, 2, 2)), col = "dodgerblue", lwd = 2,
     xlab = "mean number of tools", ylab = "Density", main = "exp(Bc)")
legend("topright", legend = "Bc ~ LN(2, 2)", col = "dodgerblue", lwd = 2, bty = "n")
# savefig('ln2.jpg', width = 6, height = 4)

Hummm...not ideal - 1000's of tools isn't on the radar, maybe tighten up the variance a bit

In [ ]:
mean(exp(rnorm(10000, 2, .5)))

That seems low, maybe increase the mean a bit

In [ ]:
c(mean(exp(rnorm(10000, 3, .5))), exp(3))

Better, let's take a look

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4)
# Plot priors from N(3, .5)
plot(density(rlnorm(10000, 3, 0.5)), col = "dodgerblue", lwd = 2,
     xlab = "mean number of tools", ylab = "Density", main = "exp(Bc)")
legend("topright", legend = "Bc ~ LN(3, 0.5)", col = "dodgerblue", lwd = 2, bty = "n")
# savefig('ln05.jpg', width = 6, height = 4)

Ok, that seems reasonable for the intercept, but what about the slope? We'll likely use standardized log-population size, so let's see what that looks like

In [ ]:
# Number of samples
N <- 100
# Intercept prior
b0 <- rnorm(N, 3, 0.5)
# log-Population size prior
b1 <- rnorm(N, 0, 0.5)
# log-population (std)
xnew <- stdize(log(seq(100, 200000, length.out = 1000)))
# Plot predicted lines
options(repr.plot.width = 7, repr.plot.height = 5)
plot(NA, xlim = c(-1, 1), ylim = c(0, 100), xlab = 'log(population) (std)',
     ylab = 'total tools', main = "Prior predictive lines")
for (i in 1:N) lines(xnew, exp(b0[i] + b1[i]*xnew), col = adjustcolor("black", 0.2))
abline(v = 0, col = 'black')
# savefig('ppl.jpg')

Looks reasonable, so let's grab the data and run

In [ ]:
## Kline data

# Total tools - response
T <- kdata$total_tools

# log-Population size (std)
P <- stdize(log(kdata$population))
# Dummy for high-contact
tmp <- indexall(kdata$contact)
C <- tmp[[1]]; Ic <- tmp[[2]]
C

Because R and Stan index from 1, `Ic` runs 1, 2 rather than Python's 0, 1 - so `C[1]` is `'low'` and `C[2]` is `'high'`. Writing the likelihood as `poisson_log()` lets Stan apply the log link internally, which is exactly his $log(\lambda_i) = \beta_c + \beta_p log(P)$:

In [ ]:
Tools_code <- "
data {
  int<lower=0> N;
  int<lower=1> K;                        // number of contact levels
  array[N] int<lower=0> Tools;
  vector[N] zPopulation;
  array[N] int<lower=1, upper=K> Ic;
}
parameters {
  vector[K] Intercept;                   // Contact intercepts
  vector[K] logPop;                      // Contact effects of log-population size
}
transformed parameters {
  // Linear model
  vector[N] log_lambda = Intercept[Ic] + logPop[Ic].*zPopulation;
}
model {
  // Contact intercepts
  Intercept ~ normal(3, 0.5);
  // Contact effects of log-population size
  logPop ~ normal(0, 0.2);

  // Likelihood
  Tools ~ poisson_log(log_lambda);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC and LOO
  vector[N] log_lik;
  for (i in 1:N) log_lik[i] = poisson_log_lpmf(Tools[i] | log_lambda[i]);
}
"
Tools <- cmdstan_model(write_stan_file(Tools_code))

In [ ]:
trace_t <- sample_stan(Tools, list(N = length(T), K = length(C), Tools = T, zPopulation = P, Ic = Ic))

In [ ]:
# Label the indexed parameters with their contact level, as PyMC's dims would
tmp <- trace_t$summary(c("Intercept", "logPop"))
tmp$variable <- paste0(rep(c('Intercept', 'logPop'), each = length(C)), '[', C, ']')
tmp

So it looks like log(population) size matters for low contact places, but not for high.

To illustrate something key about parameters and overfitting in the Poisson context, let's also run an intercept-only model:

In [ ]:
Tools_null_code <- "
data {
  int<lower=0> N;
  array[N] int<lower=0> Tools;
}
parameters {
  real Intercept;                        // Contact intercepts
}
model {
  // Contact intercepts
  Intercept ~ normal(3, 0.5);

  // Likelihood
  Tools ~ poisson_log(Intercept);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC and LOO
  vector[N] log_lik;
  for (i in 1:N) log_lik[i] = poisson_log_lpmf(Tools[i] | Intercept);
}
"
Tools_null <- cmdstan_model(write_stan_file(Tools_null_code))

In [ ]:
trace_n <- sample_stan(Tools_null, list(N = length(T), Tools = T))

In [ ]:
trace_n$summary("Intercept")

In [ ]:
# Calculate tools loo
# (no equivalent of pm.compute_log_likelihood() is needed - the generated quantities
#  block already stored log_lik for every observation at every iteration)
tools_loo <- trace_t$loo()
tools_loo

In [ ]:
loo_compare(list("Tools - logPop" = tools_loo, "Tools - null" = trace_n$loo()))

We can see the log-population model does better than the null - but also that there are loo-based warnings. Where are thse coming from? We can plot the data with our model, scaling the point sizes by their Pareto-k values to see where the issues are (remember the chimps?)

(`loo_compare()` reports differences in `elpd_loo`, the log-score, so *higher* is better and the best model is listed first with a difference of zero - the opposite sign convention to the deviance scale.)

In [ ]:
# Shade a credible band from a matrix of predictive draws (rows = draws, columns = x values)
plot_band <- function(x, ymat, prob = 0.94, col = "blue"){
    lo <- apply(ymat, 2, quantile, probs = (1 - prob)/2)
    hi <- apply(ymat, 2, quantile, probs = 1 - (1 - prob)/2)
    polygon(c(x, rev(x)), c(lo, rev(hi)), col = adjustcolor(col, 0.2), border = NA)
}

In [ ]:
## Predicted values from the model using Stan

# Number of values
N <- 10
# Range of standardized population
xnew <- seq(min(P), max(P), length.out = N)

Stan has no equivalent of `pm.Data(..., mutable=True)` and `pm.set_data()`, so instead of swapping the data out from under a fitted model we take the posterior draws for the intercepts and slopes and push the counterfactual population values through the model ourselves. Each row is a posterior draw, each column one of the new population values:

In [ ]:
# Posterior draws of the contact intercepts and slopes (rows = draws, columns = contact levels)
b0_ <- trace_t$draws("Intercept", format = "matrix")
b1_ <- trace_t$draws("logPop", format = "matrix")
ilow <- which(C == 'low'); ihigh <- which(C == 'high')

# Low contact predictions
lam0 <- sapply(xnew, function(x) exp(b0_[, ilow] + b1_[, ilow]*x))
ynew0 <- matrix(rpois(length(lam0), lam0), nrow = nrow(lam0))
# High contact predictions
lam1 <- sapply(xnew, function(x) exp(b0_[, ihigh] + b1_[, ihigh]*x))
ynew1 <- matrix(rpois(length(lam1), lam1), nrow = nrow(lam1))

In [ ]:
# Grab trendlines
ymu0 <- colMeans(ynew0)
ymu1 <- colMeans(ynew1)

In [ ]:
# Grab pointwise k values
k <- tools_loo$diagnostics$pareto_k

In [ ]:
## Plot model fits with pareto k values
options(repr.plot.width = 12, repr.plot.height = 6)
par(mfrow = c(1, 2))

# Point size scaled to pareto k
psize <- k/max(k)*3

# Low contact islands
indx <- Ic == ilow

# = = = = = = = = = = = = = = = = Plot standardized scale
plot(NA, xlim = range(xnew), ylim = c(0, 130), xlab = "log(Population) (std)", ylab = "Total tools")

# Plot low-connected fit
plot_band(xnew, ynew0)
lines(xnew, ymu0, lty = 2, col = adjustcolor("blue", 0.7), lwd = 2)

# Plot highly-connected fit
plot_band(xnew, ynew1)
lines(xnew, ymu1, col = adjustcolor("blue", 0.7), lwd = 2)

# Plot names and k values
mask <- k > 0.5
text(P[mask] - 0.2, T[mask] + 4, paste0(kdata$culture[mask], "(", round(k[mask], 2), ")"), cex = 0.6, xpd = TRUE)

# Plot low-connected data
points(P[indx], T[indx], cex = psize[indx], col = "black", lwd = 1)
# Plot highly-connected data
points(P[!indx], T[!indx], cex = psize[!indx], pch = 16, col = adjustcolor("dodgerblue", 0.8))

# Make pretty
legend("topleft", legend = c("Low contact expected", "High contact expected", "low contact", "high contact"),
       lty = c(2, 1, NA, NA), pch = c(NA, NA, 1, 16), col = c("blue", "blue", "black", "dodgerblue"),
       ncol = 2, cex = 0.6, bty = "n")

# = = = = = = = = = = = = = = = = Plot natural scale
# Undstandardize prediction scale
xnew2 <- exp(xnew*sd(log(kdata$population)) + mean(log(kdata$population)))

plot(NA, xlim = c(0, 300000), ylim = c(0, 130), xlab = "Population", ylab = "Total tools")

# Plot low-connected fit
plot_band(xnew2, ynew0)
lines(xnew2, ymu0, lty = 2, col = adjustcolor("blue", 0.7), lwd = 2)

# Plot highly-connected fit
plot_band(xnew2, ynew1)
lines(xnew2, ymu1, col = adjustcolor("blue", 0.7), lwd = 2)

# Plot low-connected data
points(kdata$population[indx], T[indx], cex = psize[indx], col = "black", lwd = 1)
# Plot high-connected data
points(kdata$population[!indx], T[!indx], cex = psize[!indx], pch = 16, col = adjustcolor("dodgerblue", 0.8))

legend("topleft", legend = c("Low contact mean", "High contact mean", "low contact", "high contact"),
       lty = c(2, 1, NA, NA), pch = c(NA, NA, 1, 16), col = c("blue", "blue", "black", "dodgerblue"),
       ncol = 2, cex = 0.6, bty = "n")

# savefig('modelfits.jpg', width = 12, height = 6)
par(mfrow = c(1, 1))
options(repr.plot.width = 7, repr.plot.height = 5)

With a $N(0, 0.2)$ prior for the slopes, we can see that the high contact mean trend falls below that of Hawaii - this is a problem because low-contact Hawaii has lots of tools due to high population, so we'd expect the counterfactual high-contact Hawaii to have even more tools, but the prediction says less. This is due, in part to the fact that our standardized model isn't anchored on zero for total tools at zero total population size. We can do better than this.

*Scientific model*

Based on domain knowledge - i.e. our understanding of the system under study - it should make sense that innovation in tool development increases with population size but with diminishing returns; eventually each additional person will add less innovation. Also cultures tend to discard tools over time, replacing them with new ones. These two processes can be represented by

$$
\Delta T = \alpha P^{\beta} - \gamma T
$$

where the change in tool number per time step ($\Delta T$) is equal to some increase ($\alpha$) proportional to population size, with diminishing returns ($\beta$), and the per-unit-time rate of tool loss ($\gamma$). If we then set $\Delta T=0$, we'll get the equilibrium tool set size


$$
\hat{T} = \frac{\alpha P^{\beta}}{\gamma}.
$$

Which we can encode this into a statistical model as

$$
\begin{align}
T_i \sim & Poisson(\lambda_i) \\
\lambda_i = & \frac{\alpha P^{\beta}}{\gamma}
\end{align}
$$

Note that the link function is gone now, because we have a mechanistic model (science!).

In [ ]:
# Population size actual counts
p <- kdata$population

In [ ]:
SciTools_code <- "
data {
  int<lower=0> N;
  int<lower=1> K;                        // number of contact levels
  array[N] int<lower=0> Tools;
  vector[N] Population;
  array[N] int<lower=1, upper=K> Ic;
}
parameters {
  vector[K] innovation;                  // Innovation increase with population
  vector<lower=0>[K] dim_returns;        // Diminishing returns
  real<lower=0> tool_loss;               // Tool loss rate
}
transformed parameters {
  // Scientific model
  vector[N] lambda;
  for (i in 1:N)
    lambda[i] = exp(innovation[Ic[i]])*pow(Population[i], dim_returns[Ic[i]])/tool_loss;
}
model {
  // Innovation increase with population
  innovation ~ normal(.5, 1);
  // Diminishing returns
  dim_returns ~ exponential(2);
  // Tool loss rate
  tool_loss ~ exponential(.5);

  // Likelihood
  Tools ~ poisson(lambda);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC and LOO
  vector[N] log_lik;
  for (i in 1:N) log_lik[i] = poisson_lpmf(Tools[i] | lambda[i]);
}
"
SciTools <- cmdstan_model(write_stan_file(SciTools_code))

In [ ]:
trace_st <- sample_stan(SciTools, list(N = length(T), K = length(C), Tools = T, Population = p, Ic = Ic))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)
mcmc_combo(trace_st$draws(c("innovation", "dim_returns", "tool_loss")), combo = c("dens_overlay", "trace"))
# ggsave('sciencetrace.jpg', dpi = 300)

Ok, things looking good here on the estimation side - how about plotting our model against our observed data

In [ ]:
## Predicted values from the model using Stan

# Number of values
N <- 10
# Range of positive standardized population
xnew <- round(seq(min(p), max(p), length.out = N))

# Posterior draws (rows = draws, columns = contact levels)
a_ <- trace_st$draws("innovation", format = "matrix")
b_ <- trace_st$draws("dim_returns", format = "matrix")
g_ <- as.vector(trace_st$draws("tool_loss"))

# Low contact predictions
lam0 <- sapply(xnew, function(x) exp(a_[, ilow])*x^b_[, ilow]/g_)
ynew0 <- matrix(rpois(length(lam0), lam0), nrow = nrow(lam0))
# High contact predictions
lam1 <- sapply(xnew, function(x) exp(a_[, ihigh])*x^b_[, ihigh]/g_)
ynew1 <- matrix(rpois(length(lam1), lam1), nrow = nrow(lam1))

# Grab trendlines
ymu0 <- colMeans(ynew0)
ymu1 <- colMeans(ynew1)

In [ ]:
## Plot model fits with pareto k values
options(repr.plot.width = 7, repr.plot.height = 6)

# Point size scaled to pareto k
psize <- k/max(k)*3

# = = = = = = = = = = = = = = = = Plot natural scale
# Undstandardize prediction scale
xnew2 <- xnew

plot(NA, xlim = c(-10000, 300000), ylim = c(0, 130), xlab = "Population", ylab = "Total tools")

# Plot low-connected fit
plot_band(xnew2, ynew0)
lines(xnew2, ymu0, lty = 2, col = adjustcolor("blue", 0.7), lwd = 2)

# Plot highly-connected fit
plot_band(xnew2, ynew1)
lines(xnew2, ymu1, col = adjustcolor("blue", 0.7), lwd = 2)

# Plot low-connected data
points(kdata$population[indx], T[indx], cex = psize[indx], col = "black", lwd = 1)
# Plot high-connected data
points(kdata$population[!indx], T[!indx], cex = psize[!indx], pch = 16, col = adjustcolor("dodgerblue", 0.8))

legend("topleft", legend = c("Low contact mean", "High contact mean", "low contact", "high contact"),
       lty = c(2, 1, NA, NA), pch = c(NA, NA, 1, 16), col = c("blue", "blue", "black", "dodgerblue"),
       ncol = 2, cex = 0.6, bty = "n")

# savefig('sciencefits.jpg', width = 7, height = 6)
options(repr.plot.width = 7, repr.plot.height = 5)

## Gamma-Poisson mixture (aka Negative Binomial)

While Poisson distributions abound in nature, the hard scaling of the mean and variance being equal to $\lambda$ means that it can fail in cases where the variance is either greater or less than we'd expect. In these cases we can turn to the [Negative Binomial](https://en.wikipedia.org/wiki/Negative_binomial_distribution), which can be thought of as (and often arises from) a mixture of Poisson distribtuions whose rates are [Gamma distributed](https://en.wikipedia.org/wiki/Gamma_distribution). This is, in my view, a much more intuitive representation than the Negative Binomial's namesake definition, which relates to the expected number of times you something fails until you see a success.

In the case of our scientific model above, we can use the Negative Binomial in place of the Poisson to allow for *overdispersion* - the rate at which the variance departs from the mean as it increases, namely:

$$
var_{NB} = \lambda+\lambda^{2}/\phi
$$

where $\phi$ conrols the rate of departure from the $mean=variance$ relationship of the Poisson. As $\phi$ increases, the NB converges toward a Poisson. 

Let's apply this model to the Oceania data and see what happens:

In [ ]:
SciTools_NB_code <- "
data {
  int<lower=0> N;
  int<lower=1> K;                        // number of contact levels
  array[N] int<lower=0> Tools;
  vector[N] Population;
  array[N] int<lower=1, upper=K> Ic;
}
parameters {
  vector[K] iRate;                       // Innovation increase with population
  vector<lower=0>[K] dReturns;           // Diminishing returns
  real<lower=0> lRate;                   // Tool loss rate
  real<lower=0> phi;                     // Gamma mixture parameter
}
transformed parameters {
  // Scientific model
  vector[N] lambda;
  for (i in 1:N)
    lambda[i] = exp(iRate[Ic[i]])*pow(Population[i], dReturns[Ic[i]])/lRate;
}
model {
  // Innovation increase with population
  iRate ~ normal(1, 1);
  // Diminishing returns
  dReturns ~ exponential(2);
  // Tool loss rate
  lRate ~ exponential(.5);

  // Gamma mixture parameter
  phi ~ gamma(0.01, 0.01);

  // Likelihood
  Tools ~ neg_binomial_2(lambda, phi);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC and LOO
  vector[N] log_lik;
  for (i in 1:N) log_lik[i] = neg_binomial_2_lpmf(Tools[i] | lambda[i], phi);
}
"
SciTools_NB <- cmdstan_model(write_stan_file(SciTools_NB_code))

In [ ]:
trace_stnb <- sample_stan(SciTools_NB, list(N = length(T), K = length(C), Tools = T, Population = p, Ic = Ic))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 7)
mcmc_combo(trace_stnb$draws(c("iRate", "dReturns", "lRate", "phi")), combo = c("dens_overlay", "trace"))

In [ ]:
## Predicted values from the model using Stan

# Number of values
N <- 10

# Posterior draws (rows = draws, columns = contact levels)
a_ <- trace_stnb$draws("iRate", format = "matrix")
b_ <- trace_stnb$draws("dReturns", format = "matrix")
g_ <- as.vector(trace_stnb$draws("lRate"))
phi_ <- as.vector(trace_stnb$draws("phi"))

# Low contact predictions
lam0 <- sapply(xnew, function(x) exp(a_[, ilow])*x^b_[, ilow]/g_)
ynew0 <- matrix(rnbinom(length(lam0), mu = lam0, size = phi_), nrow = nrow(lam0))
# High contact predictions
lam1 <- sapply(xnew, function(x) exp(a_[, ihigh])*x^b_[, ihigh]/g_)
ynew1 <- matrix(rnbinom(length(lam1), mu = lam1, size = phi_), nrow = nrow(lam1))

# Grab trendlines
ymu0 <- colMeans(ynew0)
ymu1 <- colMeans(ynew1)

In [ ]:
## Plot model fits with pareto k values
options(repr.plot.width = 7, repr.plot.height = 6)

# Point size scaled to pareto k
psize <- k/max(k)*3

# = = = = = = = = = = = = = = = = Plot natural scale
# Undstandardize prediction scale
xnew2 <- xnew

plot(NA, xlim = c(-10000, 300000), ylim = c(0, 130), xlab = "Population", ylab = "Total tools")

# Plot low-connected fit
plot_band(xnew2, ynew0)
lines(xnew2, ymu0, lty = 2, col = adjustcolor("blue", 0.7), lwd = 2)

# Plot highly-connected fit
plot_band(xnew2, ynew1)
lines(xnew2, ymu1, col = adjustcolor("blue", 0.7), lwd = 2)

# Plot low-connected data
points(kdata$population[indx], T[indx], cex = psize[indx], col = "black", lwd = 1)
# Plot high-connected data
points(kdata$population[!indx], T[!indx], cex = psize[!indx], pch = 16, col = adjustcolor("dodgerblue", 0.8))

legend("topleft", legend = c("Low contact mean", "High contact mean", "low contact", "high contact"),
       lty = c(2, 1, NA, NA), pch = c(NA, NA, 1, 16), col = c("blue", "blue", "black", "dodgerblue"),
       ncol = 2, cex = 0.6, bty = "n")

# savefig('sciencefits_nb.jpg', width = 7, height = 6)
options(repr.plot.width = 7, repr.plot.height = 5)

# Poisson exposure rates

A key aspect of Poisson (or NB) mean parameters is that they can be thought of as rates - which means that they can be thought of as the count of events per unit exposure. Exposure can be any kind of standardization - per capita, per unit area, per kg, whatever - and to represent it, we can add a simple offset to our model, namely:


$$
\begin{align}
y_i \sim & Poisson(\lambda_i) \\
log(\lambda_i) = & log(\frac{\mu_i}{\tau_i}) = \beta_0 + \beta_1x_i
\end{align}
$$

is equivent to 

$$
\begin{align}
y_i \sim & Poisson(\mu_i) \\
log(\mu_i) = & log(\tau_i) + \beta_0 + \beta_1x_i
\end{align}
$$

where $\mu_i$ is the expected count and $\tau_i$ is the exposure. This often comes up so it is important to know about.